# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [47]:
# Escribe tu código aquí para limpiar y enriquecer los datos
# Librerías necesarias
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

In [48]:
# Cargar el conjunto de datos de entrenamiento (train)
df = pd.read_csv("../data/interim/train_set.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-118.39,33.88,33.0,2543.0,439.0,1098.0,416.0,5.9683,495500.0,<1H OCEAN
1,-122.44,37.76,50.0,2589.0,569.0,945.0,544.0,5.2519,376600.0,NEAR BAY
2,-118.19,33.82,11.0,872.0,203.0,422.0,221.0,4.6364,156300.0,NEAR OCEAN
3,-118.14,34.16,30.0,2598.0,757.0,2869.0,769.0,2.1377,142300.0,<1H OCEAN
4,-117.31,33.16,17.0,1704.0,263.0,781.0,281.0,5.6605,224400.0,NEAR OCEAN


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13209 entries, 0 to 13208
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           13209 non-null  float64
 1   latitude            13209 non-null  float64
 2   housing_median_age  13209 non-null  float64
 3   total_rooms         13209 non-null  float64
 4   total_bedrooms      13069 non-null  float64
 5   population          13209 non-null  float64
 6   households          13209 non-null  float64
 7   median_income       13209 non-null  float64
 8   median_house_value  13209 non-null  float64
 9   ocean_proximity     13209 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.0 MB


1. Solventar Problemas de calidad

- Solventar problemas de sensibilidad: Existen outliers en algunas columnas, sin embargo, son datos importantes y necesarios, por lo que se mantendrán los valores.

- Solventar problemas de precicisión: En el análisis de duplicados, no existen duplicados explícitos como tampoco existen duplicados engañosos, por lo que esta sección no modificará datos.

- Solventar problemas de completitud: considerando que es MCAR, se usará MICE para llenar los datos faltantes.

In [50]:
# Solventar problemas de completitud, para mecanismo MCAR.
# Identificar filas con datos faltantes
datos_con_faltantes = df[df.isnull().any(axis=1)]
datos_con_faltantes.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
79,-121.08,39.22,30.0,2188.0,NaN,1033.0,437.0,2.1419,105200.0,INLAND
176,-122.23,38.17,45.0,350.0,NaN,225.0,72.0,1.8942,216700.0,NEAR BAY
238,-117.28,34.26,18.0,3895.0,NaN,1086.0,375.0,3.3672,133600.0,INLAND
352,-117.91,33.76,20.0,4413.0,NaN,4818.0,1063.0,2.8594,215100.0,<1H OCEAN
463,-118.30,33.73,42.0,1731.0,NaN,866.0,403.0,2.7451,255400.0,NEAR OCEAN


In [51]:
print("Valores faltantes antes:", df["total_bedrooms"].isnull().sum())

Valores faltantes antes: 140


In [52]:
# Imputación de datos faltantes en la columna "total_bedrooms" usando MICE
imputer = IterativeImputer(max_iter=10,random_state=42)
columnas_numericas = df.select_dtypes(include="number").columns
df[columnas_numericas] = imputer.fit_transform(df[columnas_numericas])

In [53]:
# Verificación de la imputación
print("Valores faltantes después:", df["total_bedrooms"].isnull().sum())

Valores faltantes después: 0


2. Codificación Categórica

In [54]:
# verificar valores únicos en ocean_proximity
df["ocean_proximity"].unique()

<StringArray>
['<1H OCEAN', 'NEAR BAY', 'NEAR OCEAN', 'INLAND', 'ISLAND']
Length: 5, dtype: str

In [55]:
# valor promedio de median_house_value en las diferentes categorías de ocean_proximity
df.groupby("ocean_proximity")["median_house_value"].mean()

ocean_proximity
<1H OCEAN     240546.913585
INLAND        124093.157392
ISLAND        450000.000000
NEAR BAY      261093.544842
NEAR OCEAN    246009.197080
Name: median_house_value, dtype: float64

In [56]:
# ordenar de mayor a menor el valor promedio de median_house_value en las diferentes categorías de ocean_proximity
df.groupby("ocean_proximity")["median_house_value"].mean().sort_values(ascending=False)

ocean_proximity
ISLAND        450000.000000
NEAR BAY      261093.544842
NEAR OCEAN    246009.197080
<1H OCEAN     240546.913585
INLAND        124093.157392
Name: median_house_value, dtype: float64

Para la trasnformación de las variables categóricas se usa OrdinalEncoder, sin especificar un valor, de modo que la librería encuentre los valores adecuados, ya que no se pudo encontrar un patrón adecuado de clasificación.

In [57]:
#Considerando la longitud y latitud, se puede inferir que las categorías de ocean_proximity tienen un orden lógico, relacionado con la cercanía al océano. Por lo tanto, se puede usar ordinal encoding para convertir los datos de ocean_proximity a números.
encoder = OrdinalEncoder()
df["ocean_proximity"] = encoder.fit_transform(df[["ocean_proximity"]])
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-118.39,33.88,33.0,2543.0,439.0,1098.0,416.0,5.9683,495500.0,0.0
1,-122.44,37.76,50.0,2589.0,569.0,945.0,544.0,5.2519,376600.0,3.0
2,-118.19,33.82,11.0,872.0,203.0,422.0,221.0,4.6364,156300.0,4.0
3,-118.14,34.16,30.0,2598.0,757.0,2869.0,769.0,2.1377,142300.0,0.0
4,-117.31,33.16,17.0,1704.0,263.0,781.0,281.0,5.6605,224400.0,4.0


3. Enriquecimiento (Feature Engineering)

 - métricas útiles:
    - `rooms_per_household = total_rooms / households`
    - `bedrooms_per_room = total_bedrooms / total_rooms`
    - `population_per_household = population / households`

In [58]:
# Creamos 3 variables nuevas más informativas
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

print("Nuevas variables:")
df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe()

Nuevas variables:


,rooms_per_household,bedrooms_per_room,population_per_household
count,13209.000000,13209.000000,13209.000000
mean,5.440991,0.212618,3.010973
std,2.629979,0.056957,4.966110
min,0.888889,-0.762957,0.692308
25%,4.452055,0.175501,2.433657
50%,5.233100,0.203390,2.823838
75%,6.058480,0.239314,3.288618
max,141.909091,1.000000,502.461538


In [59]:
# Verificar la correlación entre todas las variables y median_house_value
correlacion = df.corr()
print("Correlación entre todas las variables y median_house_value:")
print(correlacion["median_house_value"].sort_values(ascending=False))

Correlación entre todas las variables y median_house_value:
median_house_value          1.000000
median_income               0.689805
rooms_per_household         0.140833
total_rooms                 0.140251
housing_median_age          0.107478
ocean_proximity             0.075363
households                  0.075054
total_bedrooms              0.058932
population                 -0.018331
population_per_household   -0.033324
longitude                  -0.052017
latitude                   -0.139748
bedrooms_per_room          -0.257487
Name: median_house_value, dtype: float64


4. Escalado de variables

In [60]:
# dividir el conjunto de datos en características (X) y variable objetivo (y)
X_train = df.drop("median_house_value", axis=1)
y_train = df["median_house_value"]

# Escalar las variables numéricas para mejorar el rendimiento de los modelos de machine learning
scaler = StandardScaler()
numerical_features = X_train.select_dtypes(include="number").columns
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])